# Project Config

In [2]:
from pathlib import Path
import subprocess

In [3]:
def make_dir(dir_path: Path):
    dir_path.mkdir(parents=True, exist_ok=True)

def file_structure(assets_dir: Path):
    make_dir(assets_dir)

    input_dir = assets_dir / "input"
    make_dir(input_dir)
    queue_dir = assets_dir / "queue"
    make_dir(queue_dir)
    history_dir = assets_dir / "history"
    make_dir(history_dir)

    return input_dir, queue_dir, history_dir
    
 
    csv_dir = history_dir / "csv"
    make_dir(csv_dir)

def clean_dir(dir_path: Path):
    for file in dir_path.glob("*"):
        if file.is_dir():
            clean_dir(file)
        else:
            file.unlink()
    dir_path.rmdir()


In [4]:
project_root = Path('__file__').parent
data_dir  = Path(project_root / "note_assets")

input_dir, queue_dir, history_dir = file_structure(data_dir)

In [ ]:
clean_dir(data_dir)

# load data into input file

In [5]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("VLM2Vec/MSR-VTT", "test_1k")
print(ds)


/Users/keimatsui/Documents/dev/videoLLM/.venv-vlm/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    test: Dataset({
        features: ['video_id', 'video', 'caption', 'source', 'category', 'url', 'start time', 'end time', 'id'],
        num_rows: 1000
    })
})


In [6]:
def download_video(video_id: str, output_dir: Path, url: str):
    try:
        subprocess.run([
            "yt-dlp",
            "--no-playlist",
            "-t", "mp4",
            "-o", str(output_dir / f"{video_id}.mp4"),
            url
            ]
        )
    except subprocess.CalledProcessError as e:
        print(f"Error downloading video {video_id}: {e}")

count = 0
MAX_COUNT = 20
for video_id, url in zip(ds['test']['video_id'], ds['test']['url']):
    download_video(video_id, input_dir, url)
    count += 1
    if count > MAX_COUNT:
        break

print(f"Downloaded {len(list(input_dir.glob('*.mp4')))} videos")


[youtube] Extracting URL: https://www.youtube.com/watch?v=4KRMJNBjrEs
[youtube] 4KRMJNBjrEs: Downloading webpage
[youtube] 4KRMJNBjrEs: Downloading visionos player API JSON
[youtube] 4KRMJNBjrEs: Downloading m3u8 information
[info] 4KRMJNBjrEs: Downloading 1 format(s): 137+140
[download] note_assets/input/video7020.mp4 has already been downloaded
[VideoRemuxer] Not remuxing media file "note_assets/input/video7020.mp4"; already is in target format mp4
[youtube] Extracting URL: https://www.youtube.com/watch?v=bsTJrmwyr-k
[youtube] bsTJrmwyr-k: Downloading webpage



ERROR: Interrupted by user


KeyboardInterrupt: 

In [7]:
user_query = pd.DataFrame()
user_query["query"] = pd.DataFrame(ds['test'].select_columns(["caption"]))
user_query["expected_videoid"] = pd.DataFrame(ds['test'].select_columns(["video_id"]))
print(user_query)


                                                 query expected_videoid
0           a woman creating a fondant baby and flower        video7020
1                            baseball player hits ball        video7021
2    little pet shop cat getting a bath and washed ...        video7024
3                   a naked child runs through a field        video7025
4                a man is giving a review on a vehicle        video7026
..                                                 ...              ...
995  a crowd of people sitting next to each other a...        video9909
996  a man walks towards a woman and the woman brea...        video9919
997            a group of people are dancing in a room        video9957
998  two girls are sitting in the bed with a cat an...        video9969
999  a yellow sports car with a guy speaking about ...        video9975

[1000 rows x 2 columns]


# Video / Audio Processing 

In [8]:
import subprocess
import uuid

In [9]:
def chunk_media(source: str, output_dir: str, seconds: int = 30, video_id: str = None):
    source = Path(source)
    if video_id is None:
        data_id = uuid.uuid4()
    else:
        data_id = video_id
    output = Path(output_dir)
    pattern = output / f"{data_id}_%06d{source.suffix}"
    command = [
        "ffmpeg",
        "-y",
        "-i", str(source), #input file
        "-f", "segment", #format
        "-segment_time", str(seconds), #segment time
        "-reset_timestamps", "1", #reset timestamps
        "-c", "copy", #copy
        "-segment_list", str(output / f"{data_id}_segments.csv"),
        "-segment_list_type", "csv",
        str(pattern), #output pattern
    ]
    subprocess.run(command, check=True)
    return data_id

In [22]:
for file in input_dir.glob("*.mp[34]"):
    video_id = file.stem
    chunk_media(file, queue_dir, video_id=video_id)

ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.3.34.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      61.  1.102 / 61.  1.102
  libavcodec     63.  1.102 / 63.  1.102
  libavformat    63.  1.102 / 63.  1.102
  libavdevice    63.  1.102 / 63.  1.102
  libavfilter    12.  1.102 / 12.  1.102
  libswscale     10.  1.102 / 10.  1.102
  libswresample   7.  1.102 /  7.  1.102
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'note_assets/input/video7024.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2avc1mp41
    e

# Audio to Text Processing

This section explores whether splitting the audio into shunks and transcribing it makes it faster than processing the whole audio

In [9]:
import whisper
import time

In [10]:
model = whisper.load_model("small")
audio_data = []

In [11]:
def current_time():
    return time.time()


In [24]:
# clips = sorted(list(output_dir.glob("*")))
# clip_data = {}
# print(f"Found {len(clips)} clips")
# print(clips)
# process_start = current_time()
# print(f"Start transcribing chunks at {process_start}")
# for i, clip in enumerate(clips):
#     start = current_time()
#     video_id, clip_id = clip.stem.split("_")
#     path = Path(clip)
#     result = model.transcribe(str(path), word_timestamps=True, fp16=False)
#     clip_data= {
#         "video_id": video_id,
#         "clip_id": clip_id,
#         "modality": "audio",
#         "text": result["text"]
#     }
#     audio_data.append(clip_data)
#     print(f"Transcribed chunk {i+1}/{len(clips)}: {current_time() - start} seconds")
# print(f"Total time taken: {current_time() - process_start} seconds")

Found 463 clips
[PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000000.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000001.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000002.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000003.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000004.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000005.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000006.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000007.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000008.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000009.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000010.mp4'), PosixPath('note_assets/output/222c72dd-8207-42dd-9218-696bc562cc65_000011.mp4'), PosixPath('

KeyboardInterrupt: 

In [39]:
print("Transcribing the whole video...")
process_start = current_time()
print(f"Start transcribing the whole video at {process_start}")
result = model.transcribe(str(input_dir / "vsample1.mp4"), word_timestamps=True, fp16=False)
print(f"Total time taken: {current_time() - process_start} seconds")



Transcribing the whole video...
Start transcribing the whole video at 1792146097.742368


AttributeError: 'SentenceTransformer' object has no attribute 'transcribe'

In [38]:
for segment in result["segments"]:
   print(f"Start: {segment['start']}, End: {segment['end']},{segment['text']}")


KeyError: 'segments'

# Embedder - Embedding Gemma 2 by google


In [10]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Use your Mac GPU if available; otherwise use CPU.
device = "mps" if torch.backends.mps.is_available() else "cpu"

model = SentenceTransformer(
    "google/embeddinggemma-2",
    device=device,
    model_kwargs={"torch_dtype": torch.float32},
    config_kwargs={"audio_config": None},
)

W1019 11:24:41.606000 29092 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 624/624 [00:00<00:00, 649.03it/s]


In [11]:
# video data preparation
import pandas as pd

video_metadata = []
for file in queue_dir.glob("*.csv"):
    video_id, _ = file.stem.split("_")
    df = pd.read_csv(file, header=None, names=["name", "start", "end"])
    for index, row in df.iterrows():
        video_metadata.append({
            "video_id": video_id,
            "clip_id": index,
            "modality": "video",
            "file_name": row["name"],
            "start": row["start"],
            "end": row["end"]
        })

print(video_metadata)



[{'video_id': 'video7020', 'clip_id': 0, 'modality': 'video', 'file_name': 'video7020_000000.mp4', 'start': 0.0, 'end': 36.5365}, {'video_id': 'video7020', 'clip_id': 1, 'modality': 'video', 'file_name': 'video7020_000001.mp4', 'start': 36.5365, 'end': 63.063}, {'video_id': 'video7020', 'clip_id': 2, 'modality': 'video', 'file_name': 'video7020_000002.mp4', 'start': 63.063, 'end': 91.424667}, {'video_id': 'video7020', 'clip_id': 3, 'modality': 'video', 'file_name': 'video7020_000003.mp4', 'start': 91.424667, 'end': 121.7216}, {'video_id': 'video7020', 'clip_id': 4, 'modality': 'video', 'file_name': 'video7020_000004.mp4', 'start': 121.7216, 'end': 151.184367}, {'video_id': 'video7020', 'clip_id': 5, 'modality': 'video', 'file_name': 'video7020_000005.mp4', 'start': 151.184367, 'end': 180.3802}, {'video_id': 'video7020', 'clip_id': 6, 'modality': 'video', 'file_name': 'video7020_000006.mp4', 'start': 180.3802, 'end': 210.443567}, {'video_id': 'video7020', 'clip_id': 7, 'modality': 'vide

# Vector Databse

In [12]:
from pinecone import Pinecone, SchemaBuilder
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))



### Create Index

In [18]:
schema = (
    SchemaBuilder()
    .add_dense_vector_field(name="embedding", dimension=768, metric="cosine")
    .build()
)

if not pc.indexes.exists("video-vectors"):
    print("Creating index...")
    pc.create_index("video-vectors", schema=schema)

index = pc.Index("video-vectors")

### pass vido file into embedder and insert into database vector

In [19]:
print(model.prompts)

{'query': 'task: search result | query: ', 'document': 'title: none | text: ', 'BitextMining': 'task: search result | query: ', 'Classification': 'task: classification | query: ', 'Clustering': 'task: clustering | query: ', 'CodeRetrieval': 'task: code retrieval | query: ', 'Document': 'title: none | text: ', 'FactChecking': 'task: fact checking | query: ', 'InstructionRetrieval': 'task: code retrieval | query: ', 'MultilabelClassification': 'task: classification | query: ', 'PairClassification': 'task: sentence similarity | query: ', 'QuestionAnswering': 'task: question answering | query: ', 'Reranking': 'task: search result | query: ', 'Retrieval': 'task: search result | query: ', 'Retrieval-document': 'title: none | text: ', 'Retrieval-query': 'task: search result | query: ', 'STS': 'task: sentence similarity | query: ', 'SearchQuery': 'task: search result | query: ', 'SentenceSimilarity': 'task: sentence similarity | query: ', 'Summarization': 'task: sentence similarity | query: '}

In [ ]:
for video in video_metadata:
    file_path = queue_dir / video["file_name"]
    video_id = video["video_id"]
    clip_id = video["clip_id"]
    id = f"{video_id}_{clip_id}"
    if not file_path.exists():
        continue
    video_embedding = model.encode(
        str(file_path),
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
        show_progress_bar=True,
        processing_kwargs={
            "fps": 1,
            "max_frames": 32,
            "add_timestamps": True,
            "max_soft_tokens": 280,
            "overflow_strategy": "uniform",
        }
    ).reshape(-1)

    index.documents.upsert(
        namespace="videos",
        documents=[
            {
                "_id": record_id,
                "embedding": video_embedding.tolist(),
                "video_id": str(video_id),
                "clip_id": int(clip_id),
                "modality": "video",
                "start": float(video["start"]),
                "end": float(video["end"]),
            }
        ],
    )
    
    

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:12<00:00, 12.04s/it]


PineconeValueError: Vector dict must contain an 'id' key

# Retrieval

In [30]:
def retrieve(query: str, top_k: int = 5):
    query_vector = model.encode(
        query,
        prompt_name="SearchQuery",
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).reshape(-1)
    # Cosine similarity because both sides are normalized.
    scores = embedding_matrix @ query_vector
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for index in top_indices:
        result = {
            **records[index],
            "score": float(scores[index]),
        }
        results.append(result)
    return results

In [35]:
results = retrieve(
    "the lecturer explains supervised learning",
    top_k=10,
)

for result in results:
    print(
        f"Score: {result['score']:.4f} | "
        f"Time: {result['start']:.2f}–{result['end']:.2f}s | "
        f"Clip: {result['clip']} |"
        f"Video: {result['video_path']}"
    )

Score: 0.8324 | Time: 3422.67–3452.67s | Clip: 114 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000114.mp4
Score: 0.8324 | Time: 3422.67–3452.67s | Clip: 114 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000114.mp4
Score: 0.8310 | Time: 3392.67–3422.67s | Clip: 113 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000113.mp4
Score: 0.8310 | Time: 3392.67–3422.67s | Clip: 113 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000113.mp4
Score: 0.8278 | Time: 3242.67–3272.67s | Clip: 108 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000108.mp4
Score: 0.8278 | Time: 3242.67–3272.67s | Clip: 108 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000108.mp4
Score: 0.8277 | Time: 3152.67–3182.67s | Clip: 105 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000105.mp4
Score: 0.8277 | Time: 3152.67–3182.67s | Clip: 105 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000105.mp4
Score: 0.8277 | Time: 3122.67–3152.67s | Clip: 104 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000104.mp4
Score: 0.8277 | Time: 3122.67–3152.67

# Testing & Evaluation